[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/10.%20Clase%2010/10Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=02.%20Parte%202%2F10.%20Clase%2010%2F10Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 10: Inclusión de un activo libre de riesgo en el portafolio

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Incorporar un **activo libre de riesgo** (bono) al portafolio de Markowitz.
- Derivar la **Capital Market Line** (CML) como frontera eficiente extendida.
- Comparar fronteras eficientes **con y sin bono** usando CVXPY.
- Entender la **dualidad** y los **precios sombra** del problema (Boyd & Vandenberghe, 2004, §5.1–5.6).

---

## Introducción teórica

### El activo libre de riesgo

Un activo libre de riesgo (bono, CETE, T-Bill) tiene rendimiento constante $r_f$ y varianza cero. Al incluirlo en el portafolio:
- Se extiende la matriz de covarianza con una fila/columna de ceros
- Se agrega $r_f$ al vector de rendimientos esperados
- El inversionista puede asignar parte de su capital al bono ($w_0$) y el resto a activos riesgosos ($1 - w_0$)

### Capital Market Line (CML)

La frontera eficiente se transforma de una **curva** a una **línea recta** que conecta $(0, r_f)$ con el portafolio tangente $T$:

$$
\mu_p = r_f + \frac{\mu_T - r_f}{\sigma_T} \cdot \sigma_p = r_f + S_T \cdot \sigma_p
$$

donde $S_T = (\mu_T - r_f) / \sigma_T$ es el ratio de Sharpe del portafolio tangente. La CML **domina** la frontera eficiente sin bono: para cualquier nivel de riesgo, ofrece mayor rendimiento.

### Teorema de separación (Tobin, 1958)

Todos los inversionistas, independientemente de su aversión al riesgo, mantienen la **misma combinación de activos riesgosos** (el portafolio tangente $T$). Solo varía la proporción asignada al bono. Un inversionista conservador asigna más al bono; uno agresivo, menos (o pide prestado a tasa $r_f$ para apalancarse).

## 1. Descarga de datos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.covariance as skcov
import statsmodels.api as sm

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 10)

In [ ]:
tickers = ["AAPL", "AMZN", "MSFT", "KO", "AA"]
start_date = "2025-01-01"
end_date   = "2025-03-27"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()

print(f"Activos: {list(daily_returns.columns)}")
print(f"Observaciones: {len(daily_returns)}")
daily_returns.describe()

---
## 2. Estimación robusta y extensión con bono

In [ ]:
# Estimadores robustos (solo acciones)
huber = sm.robust.scale.Huber()
mu_stocks = np.array([huber(daily_returns[col].values)[0] for col in daily_returns.columns])
Sigma_stocks = skcov.LedoitWolf().fit(daily_returns).covariance_

# Parámetros del bono
r_f = 0.045 / 252  # Tasa libre de riesgo diaria (~4.5% anual)

# Extender con bono
n_stocks = len(tickers)
n_total = n_stocks + 1
mu_with_bond = np.append(mu_stocks, r_f)
Sigma_with_bond = np.zeros((n_total, n_total))
Sigma_with_bond[:n_stocks, :n_stocks] = Sigma_stocks

tickers_with_bond = tickers + ["BOND"]

print("Rendimientos esperados (Huber) anualizados:")
for t, m in zip(tickers_with_bond, mu_with_bond):
    print(f"  {t}: {252*m:.4f}")
print(f"\nΣ extendida: {Sigma_with_bond.shape} (última fila/columna = 0)")

---
## 3. Frontera eficiente sin bono (Markowitz clásico)

In [ ]:
def efficient_frontier(mu_vec, Sigma, n_points=200):
    """Frontera eficiente con CVXPY (DCP)."""
    n = len(mu_vec)
    w = cp.Variable(n)
    target = cp.Parameter()
    risk = cp.quad_form(w, Sigma)
    prob = cp.Problem(cp.Minimize(risk),
                      [mu_vec @ w == target, cp.sum(w) == 1, w >= 0])
    means, stds = [], []
    for r in np.linspace(mu_vec.min(), mu_vec.max(), n_points):
        target.value = r
        prob.solve(solver=cp.ECOS, warm_start=True)
        if prob.status == "optimal":
            means.append(252 * r)
            stds.append(np.sqrt(252 * risk.value))
    return np.array(means), np.array(stds)

ef_means, ef_stds = efficient_frontier(mu_stocks, Sigma_stocks)
print(f"Frontera sin bono: {len(ef_means)} puntos")

---
## 4. Portafolio tangente (máximo Sharpe)

El portafolio tangente es el punto de la frontera eficiente donde la **línea desde $r_f$** es tangente a la curva. Se obtiene maximizando el ratio de Sharpe con la transformación DCP (Boyd & Vandenberghe, 2004, §4.3.2).

In [ ]:
def tangent_portfolio(mu_vec, Sigma, rf):
    """Portafolio tangente via transformación DCP."""
    n = len(mu_vec)
    excess = mu_vec - rf
    y = cp.Variable(n)
    kappa = cp.Variable(nonneg=True)
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, Sigma)),
                      [excess @ y == 1, cp.sum(y) == kappa, y >= 0])
    prob.solve(solver=cp.ECOS)
    w = y.value / kappa.value
    ret = 252 * mu_vec @ w
    risk = np.sqrt(252 * w @ Sigma @ w)
    return w, ret, risk, ret / risk

w_tangent, ret_T, risk_T, sharpe_T = tangent_portfolio(mu_stocks, Sigma_stocks, r_f)

print(f"Portafolio tangente (máximo Sharpe):")
print(f"  Rendimiento anualizado: {ret_T:.4f}")
print(f"  Riesgo anualizado:      {risk_T:.4f}")
print(f"  Ratio de Sharpe:        {sharpe_T:.4f}")
print(f"\nPesos:")
for t, w in zip(tickers, w_tangent):
    print(f"  {t}: {w:.4f}")

---
## 5. Capital Market Line (CML)

La CML es la línea recta desde $(0, r_f)$ hasta el portafolio tangente $(\sigma_T, \mu_T)$, extendida más allá (apalancamiento):

$$
\mu_{\text{CML}}(\sigma) = r_f^{\text{anual}} + \frac{\mu_T - r_f^{\text{anual}}}{\sigma_T} \cdot \sigma
$$

In [ ]:
rf_annual = r_f * 252

# CML
cml_sigmas = np.linspace(0, ef_stds.max() * 1.2, 100)
cml_returns = rf_annual + (ret_T - rf_annual) / risk_T * cml_sigmas

fig, ax = plt.subplots(figsize=(10, 7))

# Frontera sin bono
ax.plot(ef_stds, ef_means, "b-", linewidth=2, label="Frontera eficiente (solo acciones)")

# CML
ax.plot(cml_sigmas, cml_returns, "r--", linewidth=2, label=f"CML (pendiente = Sharpe = {sharpe_T:.3f})")

# Portafolio tangente
ax.scatter(risk_T, ret_T, marker="*", s=400, color="red", zorder=5,
           edgecolors="black", label=f"Portafolio tangente (S={sharpe_T:.3f})")

# Tasa libre de riesgo
ax.scatter(0, rf_annual, marker="o", s=100, color="green", zorder=5,
           label=f"r_f = {rf_annual:.2%}")

# Activos individuales
for i, ticker in enumerate(tickers):
    ret_i = 252 * mu_stocks[i]
    risk_i = np.sqrt(252) * daily_returns[ticker].std()
    ax.scatter(risk_i, ret_i, s=80, zorder=5)
    ax.annotate(ticker, (risk_i, ret_i), fontsize=10, ha="left", va="bottom")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Capital Market Line vs. Frontera Eficiente")
ax.legend(loc="upper left")
plt.tight_layout()

---
## 6. Frontera eficiente con bono

Al incluir el bono como activo adicional en la optimización, la frontera eficiente se **expande** hacia la izquierda (menor riesgo) y converge con la CML.

In [ ]:
ef_means_b, ef_stds_b = efficient_frontier(mu_with_bond, Sigma_with_bond)
print(f"Frontera con bono: {len(ef_means_b)} puntos")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(ef_stds, ef_means, "b-", linewidth=2, label="Sin bono")
ax.plot(ef_stds_b, ef_stds_b * 0 + rf_annual, ":", color="gray", alpha=0)  # invisible
ax.plot(ef_stds_b, ef_means_b, "g-", linewidth=2, label="Con bono")
ax.plot(cml_sigmas, cml_returns, "r--", linewidth=2, alpha=0.7, label="CML teórica")

ax.scatter(risk_T, ret_T, marker="*", s=400, color="red", zorder=5, edgecolors="black")
ax.scatter(0, rf_annual, marker="o", s=100, color="green", zorder=5)

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Frontera eficiente: sin bono vs. con bono vs. CML")
ax.legend()
plt.tight_layout()

---
## 7. Monte Carlo con bono

Simulamos portafolios aleatorios que incluyen el bono para visualizar cómo la nube se extiende hacia el origen.

In [ ]:
np.random.seed(42)
n_portfolios = 25000

# MC sin bono
w_mc = np.random.dirichlet(np.ones(n_stocks), n_portfolios)
ret_mc = 252 * w_mc @ mu_stocks
risk_mc = np.array([np.sqrt(252 * w @ Sigma_stocks @ w) for w in w_mc])
sharpe_mc = ret_mc / risk_mc

# MC con bono
w_mc_b = np.random.dirichlet(np.ones(n_total), n_portfolios)
ret_mc_b = 252 * w_mc_b @ mu_with_bond
risk_mc_b = np.array([np.sqrt(252 * w @ Sigma_with_bond @ w) for w in w_mc_b])
sharpe_mc_b = np.where(risk_mc_b > 0, ret_mc_b / risk_mc_b, 0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, risks, rets, sharpes, title in [
    (axes[0], risk_mc, ret_mc, sharpe_mc, "Sin bono"),
    (axes[1], risk_mc_b, ret_mc_b, sharpe_mc_b, "Con bono"),
]:
    sc = ax.scatter(risks, rets, c=sharpes, cmap="RdYlBu", s=3, alpha=0.4)
    plt.colorbar(sc, ax=ax, label="Sharpe")
    ax.plot(cml_sigmas, cml_returns, "r--", linewidth=1.5, alpha=0.7)
    ax.scatter(risk_T, ret_T, marker="*", s=200, color="red", zorder=5, edgecolors="black")
    ax.set_xlabel("Riesgo (σ)")
    ax.set_ylabel("Rendimiento")
    ax.set_title(f"MC {n_portfolios:,} portafolios — {title}")

plt.tight_layout()

---
## 8. Interpretación del teorema de separación

El teorema de separación (Tobin, 1958) implica que la **decisión de inversión se separa en dos pasos**:

1. **Encontrar el portafolio tangente** $T$ (independiente de la aversión al riesgo del inversionista).
2. **Elegir la proporción** entre $T$ y el bono según la aversión al riesgo personal.

| Inversionista | Asignación al bono ($w_0$) | Riesgo del portafolio |
|--------------|:-------------------------:|:---------------------:|
| Conservador | $w_0 > 0.5$ | Bajo |
| Moderado | $w_0 \approx 0.3$ | Medio |
| Agresivo | $w_0 \approx 0$ | Alto (= portafolio $T$) |
| Apalancado | $w_0 < 0$ (pide prestado) | Mayor que $\sigma_T$ |

In [ ]:
# Portafolios a lo largo de la CML
allocations = [0.8, 0.5, 0.3, 0.0, -0.2]  # w_0 = proporción en bono

print("Portafolios a lo largo de la CML:")
print(f"{'Tipo':<15} {'w_bono':>8} {'μ_p':>10} {'σ_p':>10} {'Sharpe':>8}")
print("-" * 55)
for w0 in allocations:
    w_risky = (1 - w0) * w_tangent
    ret_p = 252 * (w0 * r_f + (1 - w0) * mu_stocks @ w_tangent)
    risk_p = abs(1 - w0) * risk_T
    s = (ret_p - rf_annual) / risk_p if risk_p > 0 else 0
    tipo = "Conservador" if w0 > 0.5 else "Moderado" if w0 > 0.1 else "Agresivo" if w0 >= 0 else "Apalancado"
    print(f"{tipo:<15} {w0:>8.1%} {ret_p:>10.4f} {risk_p:>10.4f} {s:>8.3f}")

---

## Navegación del curso

← **Anterior**: Clase 9: Optimización Monte Carlo refinada  
→ **Siguiente**: Clase 11: Frontera eficiente con CVXPY

---

## 9. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — §4.3 (LP), §5.1–5.2 (dualidad), §5.6 (precios sombra).
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6: Mean-Variance Portfolio Theory.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Sharpe, W. F.** (1964). Capital Asset Prices. *The Journal of Finance*, 19(3), 425–442.
- **Tobin, J.** (1958). Liquidity Preference as Behavior Towards Risk. *The Review of Economic Studies*, 25(2), 65–86.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.